# REQ_110C: Warehouse Query Surface — Sunny Day Path

A guided walk through `miscope.query` — the DuckDB query surface over the Parquet warehouse.

The promise: reach stored analysis data with **plain SQL over named views**, not file globs or storage primitives. `variant_id` is already a column in every table, so a single query spans every variant in the family with no per-variant reconciliation.

```python
import miscope.query
con = miscope.query.open(family="modulo_addition_1layer")
con.df("SELECT * FROM frequency_spectrum WHERE site = 'mlp_out'")
```

Two modes share this surface:
- **Warehouse (family) mode** — `open(family=...)`. Every table becomes a view globbing across all variants' `dataviews/` dirs. Local. The day-to-day cross-variant surface (this notebook).
- **Bundle mode** — `open(root=<dir-or-url>)`. One view per flat `{root}/{table}.parquet`; works over HTTP via `httpfs` (a published bundle is queryable by URL). Noted at the end.

## 1. Open a connection

`open(family=...)` resolves the family through the sanctioned loader and registers one view per semantic/generic table, plus a unified `catalog` view. No paths in consumer code — path composition stays inside the storage primitive (`miscope.warehouse.paths`).

In [ ]:
import miscope.query

con = miscope.query.open(family="modulo_addition_1layer")

print(f"{len(con.tables())} views registered:\n")
for name in con.tables():
    print(f"  {name}")

## 2. The `catalog` view — what's stored, without reading it

The `catalog` is the `UNION ALL BY NAME` of the columnar (110-A) and tensor (110-B) descriptor planes over a shared `kind` discriminator. It's the index: every field, which analyzer produced it, how it's keyed, and where the bytes live — queryable before touching a single data row.

In [ ]:
con.df("""
    SELECT kind, COUNT(*) AS n_descriptors
    FROM catalog
    GROUP BY kind
    ORDER BY kind
""")

In [ ]:
# Which variants are materialized in this warehouse?
con.df("SELECT DISTINCT variant_id FROM catalog ORDER BY variant_id")

## 3. A single-table query, cross-variant for free

`learned_frequencies` carries `variant_id`, `epoch`, `site`, `frequency`, `is_committed`. One `WHERE` clause reads every variant at once — no loop, no path per variant. Here: the committed frequency set each variant settled on by its final recorded epoch.

In [ ]:
con.df("""
    WITH final AS (
        SELECT variant_id, MAX(epoch) AS epoch
        FROM learned_frequencies
        GROUP BY variant_id
    )
    SELECT f.variant_id,
           f.epoch,
           list(DISTINCT f.frequency ORDER BY f.frequency) AS committed_freqs
    FROM learned_frequencies f
    JOIN final USING (variant_id, epoch)
    WHERE f.is_committed
    GROUP BY f.variant_id, f.epoch
    ORDER BY f.variant_id
""")

## 4. A cross-table join — plain SQL across two views

Joins are just SQL; DuckDB resolves them over the registered views with no custom join logic in the query layer. Here we line up two independent analyzers on `(variant_id, epoch)`: the count of committed frequencies (`learned_frequencies`) against the top embedding singular value (`weight_spectra`), tracing one variant from init through grokking.

In [ ]:
con.df("""
    WITH freqs AS (
        SELECT variant_id, epoch,
               COUNT(*) FILTER (WHERE is_committed) AS n_committed
        FROM learned_frequencies
        GROUP BY variant_id, epoch
    ),
    spectra AS (
        SELECT variant_id, epoch, MAX(sv) AS top_sv
        FROM weight_spectra
        WHERE site = 'W_E'
        GROUP BY variant_id, epoch
    )
    SELECT f.variant_id, f.epoch, f.n_committed, round(s.top_sv, 3) AS w_e_top_sv
    FROM freqs f
    JOIN spectra s USING (variant_id, epoch)
    WHERE f.variant_id = 'p113_seed999_dseed598'
      AND f.epoch IN (0, 1000, 5000, 24999)
    ORDER BY f.epoch
""")

## 5. Following a tensor reference

Large arrays (basis-projection coefficient cubes, etc.) live in the **tensor** plane: the catalog stores an address and shape/dtype, not the bytes. Query the catalog to discover what's available; resolve the actual array through the variant's `tensor_catalog` accessor (110-B) — the query surface indexes tensors, it doesn't inline them.

In [ ]:
con.df("""
    SELECT analyzer, field, tensor_dtype, tensor_shape
    FROM catalog
    WHERE kind = 'tensor'
      AND variant_id = 'p113_seed999_dseed598'
      AND analyzer = 'activation_basis_projection'
    LIMIT 8
""")

## 6. `.sql()` vs `.df()`

`.df(query)` returns a pandas DataFrame (used above). `.sql(query)` returns the raw DuckDB relation — lazy, chainable, and handy when you want to push more work down to the engine before materializing. `.con` exposes the underlying connection for anything the wrapper doesn't cover.

In [ ]:
rel = con.sql("SELECT variant_id, epoch, site, frequency FROM learned_frequencies WHERE is_committed")
# Relation is lazy — refine it further before pulling a DataFrame
rel.filter("site = 'mlp_neurons' AND epoch = 24999").df()

## 7. Bundle mode (by root / URL)

The same `con.sql(...)` surface works over a flat published bundle (110-E). A local dir auto-discovers its `*.parquet` tables; a URL root needs an explicit `tables=[...]` (a remote dir can't be listed) and loads `httpfs` for range requests:

```python
# Local bundle directory
con = miscope.query.open(root="/path/to/bundle")

# Published bundle over HTTP
con = miscope.query.open(
    root="https://example.org/miscope/modadd",
    tables=["learned_frequencies", "weight_spectra"],
)
```

Queries written against the warehouse port unchanged to a bundle — same view names, same SQL.

## 8. Closing

`QueryConnection` is a context manager, so a `with` block closes the DuckDB connection on exit. Outside a `with`, call `con.close()` explicitly.

In [ ]:
con.close()

# Or scope it to a block:
with miscope.query.open(family="modulo_addition_1layer") as con:
    n = con.df("SELECT COUNT(*) AS n FROM learned_frequencies").n[0]
    print(f"learned_frequencies rows across all variants: {n}")